# 01 — Adquisición y preparación de los datos

Este notebook construye los dos conjuntos de datos principales del estudio:

1. **Universo y panel sectorial.** Lista del S&P 500 con su sector GICS, corrigiendo la reforma sectorial del 20 de marzo de 2023. La taxonomía se almacena en una copia del estado actual de los datos para que las ejecuciones futuras no se vean afectadas por reclasificaciones posteriores en la fuente.
2. **Series de precios y log-retornos semanales** (2019 – Q1 2024) y **corpus de earnings calls** del periodo 2019 – 2023 (dataset `glopardo/sp500-earnings-transcripts`).

El notebook escribe los resultados en `./data/raw/`, `./data/processed/` y `./data/matrices/`.

In [1]:
import os
from io import StringIO

import warnings

import numpy as np
import pandas as pd
import requests
import yfinance as yf
from datasets import load_dataset

In [2]:
# Rutas relativas a la raíz del proyecto
ruta_raw       = "./data/raw"
ruta_processed = "./data/processed"
ruta_matrices  = "./data/matrices"

ruta_snapshot_gics     = os.path.join(ruta_raw,      "Sectores_GICS_wikipedia_snapshot.parquet")
ruta_sectores_gics_raw = os.path.join(ruta_raw,      "Sectores_GICS_raw.parquet")
ruta_lista_tickers     = os.path.join(ruta_raw,      "Lista_Tickers.csv")
ruta_precios           = os.path.join(ruta_raw,      "Precios_acciones_2019_2024.parquet")
ruta_log_retornos      = os.path.join(ruta_raw,      "Log_Retornos_2019_2024.parquet")
ruta_corr_pearson      = os.path.join(ruta_matrices, "Matriz_Correlacion_Pearson.parquet")
ruta_transcripciones   = os.path.join(ruta_raw,      "Transcripciones_2019_2024.parquet")

for carpeta in [ruta_raw, ruta_processed, ruta_matrices]:
    os.makedirs(carpeta, exist_ok=True)

# Parámetros del estudio
START_DATE = "2019-01-01"
END_DATE   = "2024-03-31"
END_DATE_TRANSCRIPCIONES = "2023-12-31"

FECHA_REFORMA_GICS = "2023-03-20"   # reforma sectorial GICS
UMBRAL_COMPLETITUD = 0.90           # mínimo de observaciones semanales por ticker
MIN_TRANSCRIPT_LEN = 100            # mínimo de caracteres (descartar entradas corruptas)

TICKERS_DUPLICADOS   = ["GOOG", "FOX", "NWS"]   # clases de acciones repetidas
TICKERS_POST_PERIODO = ["Q", "SNDK"]            # añadidas al índice después del periodo
TICKER_RENAMES       = {"FI": "FISV"}           # Fiserv renombró su ticker en junio 2023

# Las 14 empresas afectadas por la reforma GICS del 20-03-2023.
# Recogemos el sector vigente pre-reforma para reasignarlo en los trimestres anteriores.
correcciones_pre_2023 = {
    'V':    'Information Technology',
    'MA':   'Information Technology',
    'PYPL': 'Information Technology',
    'FISV': 'Information Technology',
    'FIS':  'Information Technology',
    'GPN':  'Information Technology',
    'CPAY': 'Information Technology',
    'JKHY': 'Information Technology',
    'ADP':  'Information Technology',
    'PAYX': 'Information Technology',
    'BR':   'Information Technology',
    'TGT':  'Consumer Discretionary',
    'DG':   'Consumer Discretionary',
    'DLTR': 'Consumer Discretionary',
}

## 1. Universo del S&P 500 con clasificación GICS ajustada

Wikipedia muestra el sector GICS vigente al momento de la consulta. Si extrayésemos la información de Wikipedia de nuevo pero después de una reclasificación, el nuevo sector se propagaría hacia atrás a todo el panel temporal, introduciendo una fuga de información del futuro sobre el pasado. Para evitarlo, descargamos la tabla una sola vez y persistimos el resultado en disco; las ejecuciones posteriores cargan esa copia del estado actual, que se llama snapshot.

La reforma GICS del 20-03-2023 (que afectó a 14 empresas) la corregimos manualmente en el panel trimestral con el diccionario `correcciones_pre_2023` definido arriba.

In [3]:
if os.path.exists(ruta_snapshot_gics):
    df_empresas = pd.read_parquet(ruta_snapshot_gics)
else:
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    df_empresas = pd.read_html(StringIO(response.text))[0]
    df_empresas['Symbol'] = df_empresas['Symbol'].str.replace('.', '-', regex=False)
    df_empresas = df_empresas[['Symbol', 'Security', 'GICS Sector']].rename(
        columns={'Symbol': 'ticker', 'Security': 'company', 'GICS Sector': 'GICS_Sector_Current'}
    )
    df_empresas.to_parquet(ruta_snapshot_gics, index=False)
    print(f"Snapshot guardado en {ruta_snapshot_gics}")

print(f"Empresas en el listado: {len(df_empresas)}")
df_empresas.head()

Empresas en el listado: 503


,ticker,company,GICS_Sector_Current
0,MMM,3M,Industrials
1,AOS,A. O. Smith,Industrials
2,ABT,Abbott Laboratories,Health Care
3,ABBV,AbbVie,Health Care
4,ACN,Accenture,Information Technology


In [4]:
# Algunos tickers de Wikipedia están deslistados, fusionados o ya no cotizan.
# Comprobamos qué tickers siguen vivos pidiendo los últimos 5 días de cotización a Yahoo.

tickers_candidatos = df_empresas['ticker'].tolist()

datos_check = yf.download(
    tickers_candidatos, period="5d", progress=False, threads=True, auto_adjust=True
)['Close']

tickers_vivos   = datos_check.columns[datos_check.notna().any()].tolist()
tickers_muertos = sorted(set(tickers_candidatos) - set(tickers_vivos))

print(f"Tickers válidos en Yahoo: {len(tickers_vivos)}")
if tickers_muertos:
    print(f"Tickers descartados ({len(tickers_muertos)}): {tickers_muertos}")

df_empresas = df_empresas[df_empresas['ticker'].isin(tickers_vivos)]

Tickers válidos en Yahoo: 503


### Panel trimestral con corrección de la reforma 2023

Construimos un panel `empresa × trimestre` con el sector vigente en cada momento. La reforma GICS se ejecutó el **20-03-2023**, dentro del propio Q1-2023. Asignamos Q1-2023 al régimen *pre-reforma* por dos motivos:

1. Las earnings calls de Q1-2023 se publican en abril–mayo y discuten operaciones de un trimestre transcurrido mayoritariamente  bajo la clasificación antigua (≈ 78 de 90 días). La narrativa textual corresponde al sector pre-reforma.
2. Imputar el sector *post-reforma* a Q1-2023 introduciría información posterior a la mayor parte del periodo evaluado.

Por tanto, se cambia a la nueva clasificación a partir de Q2-2023.

In [5]:
fechas = pd.date_range(start='2019-03-31', end=END_DATE, freq='QE')
df_panel = pd.DataFrame({'date': fechas})
df_panel['quarter'] = df_panel['date'].dt.to_period('Q').astype(str)

# Cross-join: cada empresa aparece en cada trimestre
df_panel = df_panel.merge(df_empresas, how='cross')

# Sector POST-reforma como base; luego corregimos hacia atrás Q1-2023 y anteriores
df_panel['gics_sector'] = df_panel['GICS_Sector_Current']

for ticker, sector_pre in correcciones_pre_2023.items():
    mask = (df_panel['ticker'] == ticker) & (df_panel['quarter'] <= "2023Q1")
    df_panel.loc[mask, 'gics_sector'] = sector_pre

df_panel = df_panel[['date', 'quarter', 'ticker', 'company', 'gics_sector']]
df_panel.to_parquet(ruta_sectores_gics_raw, index=False)

print(f"Panel GICS: {len(df_panel):,} filas, "
      f"{df_panel['ticker'].nunique()} empresas, "
      f"{df_panel['quarter'].nunique()} trimestres.")
df_panel.head()

Panel GICS: 10,563 filas, 503 empresas, 21 trimestres.


,date,quarter,ticker,company,gics_sector
0,2019-03-31,2019Q1,MMM,3M,Industrials
1,2019-03-31,2019Q1,AOS,A. O. Smith,Industrials
2,2019-03-31,2019Q1,ABT,Abbott Laboratories,Health Care
3,2019-03-31,2019Q1,ABBV,AbbVie,Health Care
4,2019-03-31,2019Q1,ACN,Accenture,Information Technology


## 2. Precios semanales, log-retornos y correlación

Descargamos precios semanales ajustados de Yahoo Finance para el periodo del estudio. Aplicamos dos exclusiones manuales documentadas en la memoria antes de la descarga:

- **Clases de acciones repetidas**: `GOOG` (se mantiene `GOOGL`), `FOX`, `NWS`.
- **Tickers añadidos al índice después del periodo**: `Q`, `SNDK`.

A la salida, descartamos también las empresas con menos del 90% de observaciones semanales (filtro de completitud).

In [6]:
tickers_a_descargar = [t for t in tickers_vivos
                       if t not in TICKERS_DUPLICADOS + TICKERS_POST_PERIODO]
print(f"Tickers a descargar: {len(tickers_a_descargar)}")

data = yf.download(
    tickers=tickers_a_descargar,
    start=START_DATE,
    end=END_DATE,
    interval="1wk",
    auto_adjust=True,
    group_by='column',
)
df_close = data['Close']
print(f"Matriz descargada: {df_close.shape[0]} semanas × {df_close.shape[1]} tickers.")

[                       0%                       ]

Tickers a descargar: 498


[*********************100%***********************]  498 of 498 completed


Matriz descargada: 538 semanas × 498 tickers.


In [7]:
# Tickers sin ningún dato en la ventana
sin_datos = df_close.columns[df_close.isna().all()].tolist()
if sin_datos:
    print(f"Sin datos en la ventana ({len(sin_datos)}): {sin_datos}")
df_close = df_close.dropna(axis=1, how='all')

# yfinance entrega series con interval="1wk", pero la fecha de cierre no
# coincide siempre entre activos (algunos reportan en viernes, otros en el
# último día hábil del tramo). Forzamos todos al viernes con el último valor
# observado: es condición necesaria para que la matriz de correlación se
# calcule sobre fechas comunes.
df_close = df_close.resample('W-FRI').last()

# Filtro de completitud (≥ 90% de observaciones semanales)
pct_valido = df_close.count() / len(df_close)
empresas_final = pct_valido[pct_valido >= UMBRAL_COMPLETITUD].index.tolist()
descartados    = pct_valido[pct_valido <  UMBRAL_COMPLETITUD].index.tolist()
if descartados:
    print(f"Descartadas por completitud insuficiente ({len(descartados)}): {descartados}")

df_close = df_close[empresas_final]

# Segunda pasada por seguridad sobre el universo final
empresas_final = [t for t in empresas_final
                  if t not in TICKERS_DUPLICADOS + TICKERS_POST_PERIODO]
df_close = df_close[[c for c in df_close.columns if c in empresas_final]]

print(f"Universo final: {len(empresas_final)} empresas, {len(df_close)} semanas.")

Descartadas por completitud insuficiente (16): ['ABNB', 'APP', 'CARR', 'CEG', 'COIN', 'DASH', 'DDOG', 'EXE', 'GEHC', 'GEV', 'HOOD', 'KVUE', 'OTIS', 'PLTR', 'SOLV', 'VLTO']
Universo final: 482 empresas, 274 semanas.


In [8]:
# Formato largo para precios: (date, ticker, close, quarter)
df_precios = df_close.stack().reset_index()
df_precios.columns = ['date', 'ticker', 'close']
df_precios = df_precios.dropna(subset=['close'])
df_precios['quarter'] = df_precios['date'].dt.to_period('Q').astype(str)
df_precios = df_precios.sort_values(['date', 'ticker'])
df_precios.to_parquet(ruta_precios, index=False)

# Lista definitiva de tickers
pd.DataFrame(empresas_final, columns=['ticker']).to_csv(ruta_lista_tickers, index=False)
print(f"Precios guardados en {ruta_precios}")
print(f"Lista de tickers guardada en {ruta_lista_tickers}")

Precios guardados en ./data/raw\Precios_acciones_2019_2024.parquet
Lista de tickers guardada en ./data/raw\Lista_Tickers.csv


In [9]:
# Log-retornos semanales
log_ret = np.log(df_close / df_close.shift(1)).dropna(how='all')

# Matriz de correlación global de Pearson (uso descriptivo en la memoria;
# la fase de validación trabaja con Spearman para robustez frente a colas).
pearson = log_ret.corr(method='pearson')

log_ret.to_parquet(ruta_log_retornos)
pearson.to_parquet(ruta_corr_pearson)

# Curtosis temporal por activo promediada.
# La media de la curtosis intra-activo es muy superior a la de una normal (3),
# lo que justifica empíricamente el uso de Spearman.
curtosis_media = float(log_ret.kurt().mean())

print(f"Log-retornos: {log_ret.shape[0]} semanas × {log_ret.shape[1]} activos.")
print(f"Matriz de correlación: {pearson.shape[0]} × {pearson.shape[1]}")
print(f"Curtosis temporal media por activo: {curtosis_media:.4f}")

Log-retornos: 273 semanas × 482 activos.
Matriz de correlación: 482 × 482
Curtosis temporal media por activo: 8.0008


## 3. Corpus de earnings calls (Hugging Face)

Cargamos el dataset `glopardo/sp500-earnings-transcripts` y aplicamos cuatro operaciones de saneamiento:

1. **Descartar la columna `sector`** del dataset original. Refleja la clasificación GICS vigente cuando se descargó el repositorio, no la vigente en la fecha de cada earnings call. La etiqueta sectorial definitiva se construye más adelante a partir del panel GICS corregido.
2. **Re-etiquetar tickers históricos**. Fiserv cambió su ticker de `FISV` a `FI` en junio de 2023. El corpus conserva el símbolo vigente en cada fecha; lo unificamos al símbolo del universo de precios.
3. **Filtrar por universo** (intersección con la lista de tickers) y **por rango temporal** (hasta Q4-2023).
4. **Descartar transcripciones corruptas** por longitud mínima de caracteres. El umbral definitivo de calidad por número de palabras (≥ 500) se aplica más tarde.

In [10]:
warnings.filterwarnings('ignore')

dataset = load_dataset("glopardo/sp500-earnings-transcripts", split="train")
df = dataset.to_pandas()
n_hf_crudo = len(df)
print(f"Transcripciones en bruto: {n_hf_crudo:,}")

# Normalización del separador en tickers (BRK.B → BRK-B, etc.)
df['ticker'] = df['ticker'].str.replace('.', '-', regex=False)

# Eliminamos la columna `sector` del dataset fuente (ver discusión arriba)
if 'sector' in df.columns:
    df = df.drop(columns=['sector'])

# Re-etiquetado histórico de tickers
df['ticker'] = df['ticker'].replace(TICKER_RENAMES)

Transcripciones en bruto: 20,681


In [11]:
# Universo válido = lista de tickers generada en la sección anterior
tickers_validos = pd.read_csv(ruta_lista_tickers)['ticker'].tolist()

# 1. Filtro por universo
n0 = len(df)
df = df[df['ticker'].isin(tickers_validos)]
print(f"Tras filtro por universo: {len(df):,}  (eliminadas {n0 - len(df):,})")

# 2. Filtro por rango temporal
df['earnings_date'] = pd.to_datetime(df['earnings_date'])
n1 = len(df)
df = df[
    (df['earnings_date'] >= START_DATE) &
    (df['earnings_date'] <= END_DATE_TRANSCRIPCIONES)
]
print(f"Tras filtro por fechas:    {len(df):,}  (eliminadas {n1 - len(df):,})")

# Construir la columna `quarter` en formato canónico "2020Q1"
df['year'] = df['year'].astype(int)
quarter_num = df['quarter'].astype(int)
df['quarter'] = df['year'].astype(str) + "Q" + quarter_num.astype(str)

# 3. Filtro de integridad por longitud mínima de caracteres
n2 = len(df)
df = df[df['transcript'].str.len() > MIN_TRANSCRIPT_LEN]
print(f"Tras filtro de integridad: {len(df):,}  (eliminadas {n2 - len(df):,})")

Tras filtro por universo: 19,354  (eliminadas 1,327)
Tras filtro por fechas:    8,939  (eliminadas 10,415)
Tras filtro de integridad: 8,939  (eliminadas 0)


In [12]:
df_final = df[['ticker', 'company', 'earnings_date', 'quarter', 'transcript']]
df_final.to_parquet(ruta_transcripciones, index=False)

print(f"Corpus guardado en {ruta_transcripciones}")
print(f"Total de transcripciones: {len(df_final):,}")
df_final.head()

Corpus guardado en ./data/raw\Transcripciones_2019_2024.parquet
Total de transcripciones: 8,939


,ticker,company,earnings_date,quarter,transcript
20,A,Agilent Technologies,2019-02-20,2019Q1,"﻿ Operator : Good day, ladies and gentlemen, a..."
21,A,Agilent Technologies,2019-05-14,2019Q2,"﻿ Operator : Good day, ladies and gentlemen. A..."
22,A,Agilent Technologies,2019-08-14,2019Q3,"﻿ Operator : Good afternoon, and welcome to th..."
23,A,Agilent Technologies,2019-11-26,2019Q4,"﻿ Operator : Good afternoon, and welcome to th..."
24,A,Agilent Technologies,2020-02-18,2020Q1,﻿ Operator : Good afternoon and welcome to the...


## Resumen

Se muestran algunas cifras citadas en la memoria.

In [13]:
print("Universo de empresas")
print(f"  S&P 500 (snapshot Wikipedia)     : {len(tickers_candidatos):>5d}")
print(f"  Válidos en Yahoo Finance         : {len(tickers_vivos):>5d}")
print(f"  Tras exclusiones manuales        : {len(tickers_a_descargar):>5d}")
print(f"  Tras filtro de completitud (≥90%): {len(empresas_final):>5d}")

print()
print("Corpus de transcripciones")
print(f"  Hugging Face crudo               : {n_hf_crudo:>5d}")
print(f"  Tras filtro de universo,fechas e integridad : {len(df_final):>5d}")

Universo de empresas
  S&P 500 (snapshot Wikipedia)     :   503
  Válidos en Yahoo Finance         :   503
  Tras exclusiones manuales        :   498
  Tras filtro de completitud (≥90%):   482

Corpus de transcripciones
  Hugging Face crudo               : 20681
  Tras filtro de universo,fechas e integridad :  8939
